# EXP-06A — Verificação independente de implantação (Privacy Pools v1)

**Status:** piloto não executado. Testa uma transação de criação e uma de depósito referenciadas em levantamento externo, consultando **recibos e logs** no Blockscout. **Não precisa de nó archive**, não treina RF e não atribui rótulo automático.

**Uso:** `Ambiente de execução → Executar tudo`. Os resultados só serão conclusivos onde o explorador devolver dados suficientes. API indisponível → inconclusivo.

Referências: [contratos oficiais 0xbow](https://0xbow.io/blog/getting-started-with-privacy-pools), [inventário de transações candidatas](https://prifi.info/protocols/privacy-pools/) e [código do experimento no GitHub](https://github.com/BorgesGusta/ethereum-mixer-detection/blob/docs/consolidacao-oficial-2026-11-17/scripts/exp06_validar_implantacao.py).

In [ ]:
%pip -q install requests google-api-python-client google-auth-httplib2 google-auth-oauthlib
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from datetime import datetime, timezone
from pathlib import Path
import hashlib, requests, subprocess, sys, json
api = build('drive','v3')
base = Path('/content/EXP06A')
base.mkdir(parents=True, exist_ok=True)
print('Drive autorizado. Pasta local:',base)

## 1. Obter script versionado e verificar sua integridade
O código foi congelado em um commit do repositório; se os bytes não coincidirem, o notebook interrompe a execução.

In [ ]:
commit = 'c28e9ad9b129e0e0deb2433ded5509630b4ac479'
url = f'https://raw.githubusercontent.com/BorgesGusta/ethereum-mixer-detection/{commit}/scripts/exp06_validar_implantacao.py'
r=requests.get(url, timeout=40)
r.raise_for_status()
script=base/'exp06_validar_implantacao.py'
script.write_bytes(r.content)
expected_sha256='64d05561e236b91b53f57271421dce012e9268bb152af1dddb0dca4d3c961a21'
actual=hashlib.sha256(script.read_bytes()).hexdigest()
assert actual==expected_sha256, f'Script alterado: {actual}'
print('SHA-256 do script conferido:',actual)

## 2. Consultar apenas as duas transações pré-definidas
Serão gerados um JSON de evidência e um manifesto com SHA-256. **Não há coleta massiva** nem análise dos nove endereços do EXP-05.

In [ ]:
out=base/'saida'
subprocess.run([sys.executable,str(script),'--saida',str(out)],check=True)
d= json.loads((out/'verificacao_exp06a.json').read_text())
for key,val in d['transacoes'].items():
    print(key,'→',val['status_verificacao'],'HTTP',val.get('http_status'),'bloco',val.get('block_number'))
print('⚠️ Esta execução não produz rótulo científico. Confira os dados antes de qualquer classificação.')

## 3. Salvar no Drive oficial (em pasta nova)
Cria uma pasta única dentro de `results` e salva os dois arquivos. Não sobrescreve dados antigos.

In [ ]:
parent_id='1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI'
folder_name='EXP-06A_piloto_implantacao_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
folder=api.files().create(body={'name':folder_name,'mimeType':'application/vnd.google-apps.folder','parents':[parent_id]},fields='id,name').execute()
for file in sorted(out.iterdir()):
    payload=MediaFileUpload(str(file),mimetype='application/json',resumable=True)
    f=api.files().create(body={'name':file.name,'parents':[folder['id']]},media_body=payload,fields='id,name').execute()
    print('Salvo:',f['name'],f['id'])
print('PASTA EXP-06A:', 'https://drive.google.com/drive/folders/'+folder['id'])

## Depois da execução
Envie no chat **“EXP-06A executado; confira o Drive”**. Vou comparar hashes, recibos, blocos e logs. Caso a API não responda, vamos registrar o bloqueio, sem inventar uma implantação confirmada.